
# 2. Diagrama de Gantt: durada dels projectes europeus de recerca

## Objectiu

Representar temporalment un conjunt seleccionat de projectes europeus de recerca mitjançant un diagrama de Gantt.

Cada barra representarà un projecte, des de la data d'inici fins a la data de finalització. Aquesta representació permet comparar la durada dels projectes, identificar períodes de solapament i observar com es distribueixen les iniciatives en el temps.

Per facilitar la lectura, es seleccionarà una mostra reduïda de projectes amb dates vàlides.


In [1]:

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# Ruta del fitxer CSV
BASE_DIR = Path.cwd().resolve()

CSV_PATH = (
    BASE_DIR.parent
    / "dades gantt chart"
    / "project.csv"
)

# Carregar les dades
df = pd.read_csv(
    CSV_PATH,
    sep=";",
    engine="python",
    on_bad_lines="warn"
)

# Comprovar l'estructura del conjunt de dades
print("Dimensions del conjunt de dades:", df.shape)

print("\nColumnes disponibles:")
print(df.columns.tolist())

display(
    df[
        [
            "id",
            "acronym",
            "title",
            "startDate",
            "endDate",
            "frameworkProgramme"
        ]
    ].head()
)


C:\Users\roger\AppData\Local\Temp\ipykernel_14528\2888201451.py:17: ParserWarning: Skipping line 24: ';' expected after '"'

  df = pd.read_csv(
C:\Users\roger\AppData\Local\Temp\ipykernel_14528\2888201451.py:17: ParserWarning: Skipping line 44: ';' expected after '"'

  df = pd.read_csv(
C:\Users\roger\AppData\Local\Temp\ipykernel_14528\2888201451.py:17: ParserWarning: Skipping line 50: ';' expected after '"'

  df = pd.read_csv(
C:\Users\roger\AppData\Local\Temp\ipykernel_14528\2888201451.py:17: ParserWarning: Skipping line 52: ';' expected after '"'

  df = pd.read_csv(
C:\Users\roger\AppData\Local\Temp\ipykernel_14528\2888201451.py:17: ParserWarning: Skipping line 66: ';' expected after '"'

  df = pd.read_csv(
C:\Users\roger\AppData\Local\Temp\ipykernel_14528\2888201451.py:17: ParserWarning: Skipping line 73: ';' expected after '"'

  df = pd.read_csv(
C:\Users\roger\AppData\Local\Temp\ipykernel_14528\2888201451.py:17: ParserWarning: Skipping line 132: ';' expected after '"'

  df

Dimensions del conjunt de dades: (22939, 22)

Columnes disponibles:
['id', 'acronym', 'status', 'title', 'startDate', 'endDate', 'totalCost', 'ecMaxContribution', 'topics', 'ecSignatureDate', 'frameworkProgramme', 'masterCall', 'subCall', 'fundingScheme', 'nature', 'objective', 'contentUpdateDate', 'rcn', 'grantDoi', 'keywords', 'Human-validated', 'legalBasis']


,id,acronym,title,startDate,endDate,frameworkProgramme
0,101069359,SolDAC,Full spectrum SOLar Direct Air Capture & conve...,2022-09-01,2025-08-31,HORIZON
1,101069357,Photo2Fuel,Artificial PHOTOsynthesis to produce FUELs and...,2022-09-01,2025-08-31,HORIZON
2,101069586,BOLSTER,Bridging Organizations and marginalized commun...,2022-09-01,2025-08-31,HORIZON
3,101069604,GREENET,NCPS NETwork for the GREEN transition in clima...,2022-07-01,2027-12-31,HORIZON
4,101069529,SSH CENTRE,"Social Sciences and Humanities for Climate, En...",2022-09-01,2026-02-28,HORIZON


In [2]:

# Comprovar la qualitat dels camps necessaris per al diagrama de Gantt

columnes_gantt = [
    "id",
    "acronym",
    "title",
    "startDate",
    "endDate"
]

print("Valors absents als camps seleccionats:")
display(df[columnes_gantt].isna().sum())

# Convertir les dates al format datetime
df["startDate"] = pd.to_datetime(
    df["startDate"],
    errors="coerce"
)

df["endDate"] = pd.to_datetime(
    df["endDate"],
    errors="coerce"
)

# Identificar els projectes amb dates vàlides
dates_valides = (
    df["startDate"].notna()
    & df["endDate"].notna()
    & (df["endDate"] >= df["startDate"])
)

print("\nNombre de projectes llegits:", len(df))
print("Projectes amb dates vàlides:", dates_valides.sum())
print("Projectes amb dates no vàlides:", (~dates_valides).sum())

# Examinar alguns projectes vàlids
projectes_valids = df.loc[dates_valides, columnes_gantt].copy()

display(projectes_valids.head(10))


Valors absents als camps seleccionats:


id           0
acronym      0
title        0
startDate    0
endDate      0
dtype: int64


Nombre de projectes llegits: 22939
Projectes amb dates vàlides: 22939
Projectes amb dates no vàlides: 0


,id,acronym,title,startDate,endDate
0,101069359,SolDAC,Full spectrum SOLar Direct Air Capture & conve...,2022-09-01,2025-08-31
1,101069357,Photo2Fuel,Artificial PHOTOsynthesis to produce FUELs and...,2022-09-01,2025-08-31
2,101069586,BOLSTER,Bridging Organizations and marginalized commun...,2022-09-01,2025-08-31
3,101069604,GREENET,NCPS NETwork for the GREEN transition in clima...,2022-07-01,2027-12-31
4,101069529,SSH CENTRE,"Social Sciences and Humanities for Climate, En...",2022-09-01,2026-02-28
5,101069508,HighSpin,High-Voltage Spinel LNMO Silicon-Graphite Cell...,2022-09-01,2027-04-30
6,101069653,TANDEM,Transdisciplinary ANd Deliberative equity appr...,2022-09-01,2025-08-31
7,101069644,LiCORNE,Lithium recovery and battery-grade materials p...,2022-10-01,2027-03-31
8,101069612,NoVOC,Eliminating VOC from Battery manufacturing thr...,2022-09-01,2027-02-28
9,101069676,BEST,Batteries Europe Secretariat,2022-05-01,2025-04-30


In [3]:

# Seleccionar projectes del programa HORIZON

projectes_horizon = projectes_valids[
    df.loc[dates_valides, "frameworkProgramme"].eq("HORIZON").values
].copy()

# Eliminar possibles registres duplicats
projectes_horizon = projectes_horizon.drop_duplicates(subset="id")

# Ordenar els projectes per data d'inici
projectes_horizon = projectes_horizon.sort_values("startDate")

# Seleccionar una mostra de 20 projectes distribuïts temporalment
if len(projectes_horizon) > 20:
    indexos = (
        pd.Series(range(20))
        .mul((len(projectes_horizon) - 1) / 19)
        .round()
        .astype(int)
    )
    mostra_gantt = projectes_horizon.iloc[indexos].copy()
else:
    mostra_gantt = projectes_horizon.copy()

# Calcular la durada en dies
mostra_gantt["durada_dies"] = (
    mostra_gantt["endDate"] - mostra_gantt["startDate"]
).dt.days

print("Projectes seleccionats:", len(mostra_gantt))

display(
    mostra_gantt[
        ["acronym", "title", "startDate", "endDate", "durada_dies"]
    ]
)


Projectes seleccionats: 20


,acronym,title,startDate,endDate,durada_dies
4088,EUROfusion,Implementation of activities described in the ...,2021-01-01,2025-12-31,1825
2841,AI4CYBER,Trustworthy Artificial Intelligence for Cybers...,2022-09-01,2025-08-31,1095
1747,PHIDELITY,Physiological Role of Nonsense-Mediated Decay ...,2022-10-01,2025-12-31,1187
3859,SCHUMANN,Satellite Construction kit for Highly Unified ...,2023-01-01,2025-04-30,850
5357,ChOReS,Chromatin re-organization in response to repli...,2023-03-01,2028-02-29,1826
8542,RAIL4CITIES,Railway stations for green and socially inclus...,2023-07-01,2025-06-30,730
9628,DiogenX,Beta-cell recovery to counter diabetes,2023-10-01,2025-09-30,730
9797,EXSOTHyC,EXSOLUTION-BASED NANOPARTICLES FOR LOWEST COST...,2024-01-01,2026-12-31,1095
10412,PowerMiT,Leveraging the impact of gut microbes to advan...,2024-03-01,2029-02-28,1825
7549,RULIERD,Investment and EU Law: Reconceptualizing Rever...,2024-07-01,2026-06-30,729


In [4]:

# Crear el diagrama de Gantt

fig_gantt = px.timeline(
    mostra_gantt,
    x_start="startDate",
    x_end="endDate",
    y="acronym",
    color="frameworkProgramme" if "frameworkProgramme" in mostra_gantt.columns else None,
    hover_name="title",
    hover_data={
        "startDate": True,
        "endDate": True,
        "durada_dies": True,
        "acronym": False
    },
    labels={
        "acronym": "Projecte",
        "startDate": "Data d'inici",
        "endDate": "Data de finalització",
        "durada_dies": "Durada (dies)",
        "frameworkProgramme": "Programa"
    },
    title="Calendari de 20 projectes europeus de recerca"
)

# Ordenar els projectes cronològicament
fig_gantt.update_yaxes(
    autorange="reversed",
    title="Projecte"
)

fig_gantt.update_xaxes(
    title="Període d'execució",
    dtick="M12",
    tickformat="%Y"
)

fig_gantt.update_layout(
    template="plotly_white",
    height=750,
    width=1200,
    showlegend=False,
    title={
        "x": 0.5,
        "xanchor": "center"
    },
    margin=dict(l=130, r=40, t=80, b=60)
)

fig_gantt.show()


In [5]:

# Desar el diagrama de Gantt com a HTML interactiu

OUTPUT_DIR = BASE_DIR / "grafics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_path = OUTPUT_DIR / "gantt_projectes.html"

fig_gantt.write_html(
    output_path,
    include_plotlyjs=True
)

print(f"Diagrama de Gantt desat a: {output_path}")


Diagrama de Gantt desat a: D:\ESTUDIS\Master data science - UOC\1r_Semestre\Visualització de Dades\PAC2\entregable\grafics\gantt_projectes.html



## Interpretació dels resultats

El diagrama de Gantt representa el període d'execució de 20 projectes europeus de recerca. Cada barra correspon a un projecte: l'extrem esquerre indica la data d'inici i l'extrem dret, la data de finalització prevista.

La visualització permet comparar la durada dels projectes i identificar períodes en què diverses iniciatives es desenvolupen simultàniament. En la mostra seleccionada s'observa que alguns projectes coincideixen en el temps, mentre que d'altres comencen o finalitzen en moments diferents.

L'objectiu comunicatiu és facilitar la comprensió de la planificació temporal dels projectes i mostrar els solapaments entre les seves fases d'execució. El diagrama de Gantt és adequat per a aquesta finalitat perquè representa intervals temporals de manera directa i permet comparar-ne la durada.

Cal tenir en compte que es tracta d'una mostra de 20 projectes i que les dates de finalització representen els valors registrats al conjunt de dades. Per tant, la visualització no descriu necessàriament tots els projectes europeus de recerca ni confirma que les iniciatives hagin finalitzat efectivament en les dates indicades.
